In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer
from sklearn.metrics import r2_score, log_loss

from sklearn.neighbors import KNeighborsRegressor

import os
os.chdir("D:/Machine_Learning/Datasets") #fetch the dataset from the folder

In [2]:
housing = pd.read_csv('Housing.csv')
housing

,price,lotsize,bedrooms,bathrms,stories,driveway,recroom,fullbase,gashw,airco,garagepl,prefarea
0,42000.0,5850,3,1,2,yes,no,yes,no,no,1,no
1,38500.0,4000,2,1,1,yes,no,no,no,no,0,no
2,49500.0,3060,3,1,1,yes,no,no,no,no,0,no
3,60500.0,6650,3,1,2,yes,yes,no,no,no,0,no
4,61000.0,6360,2,1,1,yes,no,no,no,no,0,no
...,...,...,...,...,...,...,...,...,...,...,...,...
541,91500.0,4800,3,2,4,yes,yes,no,no,yes,0,no
542,94000.0,6000,3,2,4,yes,no,no,no,yes,0,no
543,103000.0,6000,3,2,4,yes,yes,no,no,yes,1,no
544,105000.0,6000,3,2,2,yes,yes,no,no,yes,1,no


In [3]:
# Assuming 'price' and 'lotsize' are your targets
X = housing.drop(['price', 'lotsize'], axis=1)
y = housing[['price', 'lotsize']] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26)

In [5]:
from sklearn.compose import make_column_selector

ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")

trans = ColumnTransformer(
    transformers=[("OHE", ohe, make_column_selector(dtype_include=object))], remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")


In [7]:
# Your existing OHE logic
X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)


In [11]:
Ks = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
scores = []

for k in Ks:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_trn_ohe, y_train)
    y_pred = knn.predict(X_tst_ohe)
    
    # multioutput='uniform_average' is the default
    avg_r2 = r2_score(y_test, y_pred) 
    scores.append([k, avg_r2])

df_scores = pd.DataFrame(scores, columns=['Ks', 'Avg R2 Score'])
df_scores.sort_values('Avg R2 Score', ascending=False)

,Ks,Avg R2 Score
6,7,0.444559
9,10,0.443501
8,9,0.436579
7,8,0.436181
5,6,0.434743
4,5,0.407644
3,4,0.393345
2,3,0.347009
1,2,0.283099
0,1,0.050423


In [10]:
Ks = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
results = []

for k in Ks:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_trn_ohe, y_train)
    y_pred = knn.predict(X_tst_ohe)
    
    # Get individual scores for [price, lotsize]
    individual_r2 = r2_score(y_test, y_pred, multioutput='raw_values')
    # Get the average score
    avg_r2 = r2_score(y_test, y_pred) 
    
    results.append([k, avg_r2, individual_r2[0], individual_r2[1]])

# Creating a clean DataFrame to view results
df_res = pd.DataFrame(results, columns=['K', 'Avg_R2', 'Price_R2', 'Lotsize_R2'])
df_res.sort_values('Avg_R2', ascending=False)


,K,Avg_R2,Price_R2,Lotsize_R2
6,7,0.444559,0.652588,0.236531
9,10,0.443501,0.638885,0.248116
8,9,0.436579,0.641025,0.232133
7,8,0.436181,0.636005,0.236357
5,6,0.434743,0.641625,0.227861
4,5,0.407644,0.621187,0.194100
3,4,0.393345,0.615568,0.171121
2,3,0.347009,0.609996,0.084022
1,2,0.283099,0.590786,-0.024588
0,1,0.050423,0.334964,-0.234117
